# Top-Down Heatmap HPE (Scratch, Fast Kaggle)

Sumbu A: top-down

Sumbu B: heatmap

Notebook ini baseline edukatif untuk skripsi, fokus alur algoritmik dan evaluasi cepat.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from hpe_shared import (
    benchmark_latency,
    compute_oks_pck,
    crop_with_bbox,
    decode_heatmaps_argmax,
    gaussian_heatmap,
    get_device,
    load_coco_keypoint_samples,
    read_image_bgr,
    seed_everything,
    split_samples,
    to_tensor_rgb,
    vis_mask_from_kpts,
)

seed_everything(42)
device = get_device()
print('device:', device)

In [ ]:
# Kaggle dataset path
DATA_ROOT = Path('/kaggle/input/datasets/yanplayz08/coco-subset-for-pose-estimation')
ANN_PATH = DATA_ROOT / 'annotations' / 'person_keypoints_train2017.json'
IMG_DIR = DATA_ROOT / 'train2017'

MAX_SAMPLES = 3000
TRAIN_BS = 32
EPOCHS = 3
LR = 1e-3
IN_H, IN_W = 256, 192
HM_H, HM_W = 64, 48

samples = load_coco_keypoint_samples(str(ANN_PATH), str(IMG_DIR), max_samples=MAX_SAMPLES, min_labeled_kpt=5)
train_samples, val_samples = split_samples(samples, val_ratio=0.2, seed=42)
print('samples:', len(samples), 'train:', len(train_samples), 'val:', len(val_samples))

In [ ]:
class TopDownPoseDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = read_image_bgr(s.image_path)
        crop, crop_box = crop_with_bbox(img, s.bbox, pad=0.2)
        if crop.size == 0:
            crop = img
            crop_box = np.array([0, 0, img.shape[1], img.shape[0]], dtype=np.float32)

        x = to_tensor_rgb(crop, (IN_H, IN_W)).float()

        kpts = s.keypoints.copy()
        kpts[:, 0] = (kpts[:, 0] - crop_box[0]) / max(crop_box[2], 1.0)
        kpts[:, 1] = (kpts[:, 1] - crop_box[1]) / max(crop_box[3], 1.0)

        kpts[:, 0] = np.clip(kpts[:, 0], 0.0, 1.0)
        kpts[:, 1] = np.clip(kpts[:, 1], 0.0, 1.0)

        vis = vis_mask_from_kpts(kpts)
        hm = np.zeros((17, HM_H, HM_W), dtype=np.float32)
        for j in range(17):
            if vis[j] <= 0:
                continue
            cx = float(kpts[j, 0] * (HM_W - 1))
            cy = float(kpts[j, 1] * (HM_H - 1))
            hm[j] = gaussian_heatmap(HM_H, HM_W, cx, cy, sigma=1.8)

        return {
            'image': x,
            'target_hm': torch.from_numpy(hm),
            'gt_xy_norm': torch.from_numpy(kpts[:, :2].astype(np.float32)),
            'gt_vis': torch.from_numpy(vis.astype(np.float32)),
            'area': torch.tensor(float(max(1.0, s.area)), dtype=torch.float32),
        }


class TinyTopDownHeatmap(nn.Module):
    def __init__(self, num_kpt=17):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
        )
        self.head = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(128, num_kpt, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

In [ ]:
train_dl = DataLoader(TopDownPoseDataset(train_samples), batch_size=TRAIN_BS, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(TopDownPoseDataset(val_samples), batch_size=TRAIN_BS, shuffle=False, num_workers=2, pin_memory=True)

model = TinyTopDownHeatmap().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for b in train_dl:
        x = b['image'].to(device)
        y = b['target_hm'].to(device)

        pred = model(x)
        loss = F.mse_loss(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()
        running += float(loss.item())

    print(f'epoch={epoch+1} train_loss={running/max(1,len(train_dl)):.5f}')

In [ ]:
# Validation metrics: OKS, PCK, missing-joint ratio
model.eval()
oks_all, pck_all, miss_all = [], [], []

with torch.no_grad():
    for b in val_dl:
        x = b['image'].to(device)
        pred_hm = model(x)
        pred_xy_norm = decode_heatmaps_argmax(pred_hm).cpu().numpy()

        gt_xy_norm = b['gt_xy_norm'].numpy()
        gt_vis = b['gt_vis'].numpy()
        area = b['area'].numpy()

        for i in range(pred_xy_norm.shape[0]):
            pred_xy_px = np.zeros((17, 2), dtype=np.float32)
            pred_xy_px[:, 0] = pred_xy_norm[i, :, 0] * IN_W
            pred_xy_px[:, 1] = pred_xy_norm[i, :, 1] * IN_H

            gt_kpts = np.zeros((17, 3), dtype=np.float32)
            gt_kpts[:, 0] = gt_xy_norm[i, :, 0] * IN_W
            gt_kpts[:, 1] = gt_xy_norm[i, :, 1] * IN_H
            gt_kpts[:, 2] = gt_vis[i]

            pred_vis = np.ones((17,), dtype=np.float32)
            oks, pck, miss = compute_oks_pck(pred_xy_px, pred_vis, gt_kpts, area=float(area[i]), pck_alpha=0.2)
            oks_all.append(oks)
            pck_all.append(pck)
            miss_all.append(miss)

lat_in = torch.randn(1, 3, IN_H, IN_W, device=device)
lat_ms, fps = benchmark_latency(model, lat_in, iters=120)

print('Top-Down Heatmap Summary')
print('OKS mean           :', float(np.mean(oks_all) if oks_all else 0.0))
print('PCK mean           :', float(np.mean(pck_all) if pck_all else 0.0))
print('Missing joint ratio:', float(np.mean(miss_all) if miss_all else 0.0))
print('Latency mean (ms)  :', float(lat_ms))
print('FPS approx         :', float(fps))